# Increment-Swap Coupling

We introduce the term increment-swap coupling, or simply swap coupling, denoting the controlled exchange of update increments between state variables while preserving the underlying gradient or vector field components. The scalar measure of this exchange is called the coupling coefficient $C$, defined as the empirical frequency of swaps in the discrete setting and as the time-averaged exchange rate in the continuous setting.


**Definition (Continuous Swap Coupling).** Let $\mathbf{F}: \mathbb{R}^n \to \mathbb{R}^n$ be a vector field with components $F_i$. A **swap coupling** with strength $\sigma \in [0,1]$ is the vector field $\mathbf{F}^{(\sigma)}$ defined by
$$
\mathbf{F}^{(\sigma)} = \big[(1-\sigma)I + \sigma P\big]\mathbf{F},
$$
where $I$ is the identity matrix and $P$ is a permutation matrix exchanging two (or more) components. For example, in two dimensions with $P = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$,
$$
P \begin{pmatrix} F_x \\ F_y \end{pmatrix} = \begin{pmatrix} F_y \\ F_x \end{pmatrix}.
$$

The **coupling coefficient** over interval $[0,T]$ is $C_T = \frac{1}{T}\int_0^T \sigma(t)\,dt$. The vector field $\mathbf{F}^{(\sigma)}$ represents a parameterized family of vector fields indexed by the coupling strength $\sigma$.

The operator $[(1-\sigma)I + \sigma P]$ acts on the component representation of $\mathbf{F}$ in a fixed basis; it is not a geometrically invariant object. Under a change of coordinates, $P$ does not transform as a tensor, and the same physical dynamics would be represented by a different matrix in the new basis. Later we will show a solution for this.

## A notable dynamical property at $\sigma = \tfrac{1}{2}$

For the 2D case, subtract the two component equations:

$$
\dot{x} - \dot{y} = (1-2\sigma)\big(F_x(x,y) - F_y(x,y)\big).
$$

At **exactly** $\sigma = \tfrac{1}{2}$, this gives $\frac{d}{dt}(x-y) = 0$. The quantity $(x-y)$ is conserved, and the flow is confined to lines parallel to the diagonal $x=y$. The vector field becomes purely aligned with $(1,1)$.

Consequently, if the initial condition satisfies $x_0 \neq y_0$, the trajectory can never cross the diagonal $x=y$, and the system effectively reduces to one-dimensional flow along the line $x-y = \text{c}$ (a constant value).


## Discrete Swap Coupling

For numerical integration or optimization, we consider a discrete-time formulation. Let $\mathbf{x}_i \in \mathbb{R}^n$ denote the state at iteration $i \in \mathbb{Z}_{>0}$ and let $\eta > 0$ be a fixed step size. The standard forward increment is $\Delta \mathbf{x}_i = \eta \mathbf{F}(\mathbf{x}_i)$.

We introduce a binary **swap indicator** $s_i \in \{0,1\}$. The discrete swap-coupled update is
$$
\mathbf{x}_{i+1} = \mathbf{x}_i + \eta \big[(1-s_i)I + s_i P\big] \mathbf{F}(\mathbf{x}_i).
$$

Equivalently, letting $\Delta x_i = \eta F_x(\mathbf{x}_i)$ and $\Delta y_i = \eta F_y(\mathbf{x}_i)$ in the 2D case:

- **Standard step** ($s_i=0$): $x_{i+1} = x_i + \Delta x_i$, $y_{i+1} = y_i + \Delta y_i$.
- **Swapped step** ($s_i=1$): $x_{i+1} = x_i + \Delta y_i$, $y_{i+1} = y_i + \Delta x_i$.

The **empirical coupling coefficient** over $N$ steps is
$$
C_N := \frac{1}{N}\sum_{i=1}^{N} s_i.
$$

When $s_i=0$ for all $i$, we recover the standard update. When $s_i=1$, the increments of the two components are exchanged. (In the special case of gradient descent, $\mathbf{F} = -\nabla f$ and the increments are $\Delta x_i = -\eta \partial_x f$, $\Delta y_i = -\eta \partial_y f$.)

## The relation between the gradient and the swap operator

Set $\mathbf{F} = \nabla F$. In components:
$$
\mathbf{F} = \begin{pmatrix} F_x \\ F_y \end{pmatrix} = \begin{pmatrix} \frac{\partial F}{\partial x} \\[4pt] \frac{\partial F}{\partial y} \end{pmatrix}.
$$

Apply the swap matrix $P = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$:
$$
P\mathbf{F} = \begin{pmatrix} \frac{\partial F}{\partial y} \\[4pt] \frac{\partial F}{\partial x} \end{pmatrix}.
$$

Now the coupled gradient field is:
$$
\boxed{
\nabla F^{(\sigma)} = \begin{pmatrix} (1-\sigma)\frac{\partial F}{\partial x} + \sigma \frac{\partial F}{\partial y} \\[6pt] (1-\sigma)\frac{\partial F}{\partial y} + \sigma \frac{\partial F}{\partial x} \end{pmatrix}
}
$$

The finite-difference formula approximates the raw partials:

$$
\frac{\partial F}{\partial x} \approx \frac{F(x+\Delta x, y) - F(x-\Delta x, y)}{2\Delta x},
$$
$$
\frac{\partial F}{\partial y} \approx \frac{F(x, y+\Delta y) - F(x, y-\Delta y)}{2\Delta y}.
$$

These approximations give numerical values for $F_x$ and $F_y$.

## Why Increments Are Swapped, Not Variables

If we define a new function by swapping arguments,
$$
\tilde{f}(x,y) = f(y,x),
$$
then by the chain rule
$$
\frac{\partial \tilde{f}}{\partial x}(x,y) = \frac{\partial f}{\partial y}(y,x), \qquad
\frac{\partial \tilde{f}}{\partial y}(x,y) = \frac{\partial f}{\partial x}(y,x).
$$

This is not coupling. It is simply looking at the same scalar field through a mirror. The state variables themselves have been reassigned: what was called $x$ is now playing the role of $y$, and vice versa.

Increment-swap coupling leaves the evaluation point unchanged. The gradient is computed at the current state $(x_i, y_i)$ in the usual way:
$$
\Delta x_i = -\eta\, \frac{\partial f}{\partial x}(x_i, y_i), \qquad
\Delta y_i = -\eta\, \frac{\partial f}{\partial y}(x_i, y_i).
$$

The coupling then acts as a linear operator on the *vector of increments*:
$$
\begin{pmatrix} \Delta x_i^{\text{(coupled)}} \\[4pt] \Delta y_i^{\text{(coupled)}} \end{pmatrix}
= \big[(1-s_i)I + s_i P\big]
\begin{pmatrix} \Delta x_i \\[4pt] \Delta y_i \end{pmatrix}.
$$


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- style constants ---
SURFACE = "#fcfcfb"
TEXT_PRIMARY = "#1a1a19"
TEXT_SECONDARY = "#5f5e56"
GRID = "#e8e7e2"
C_F1 = "#2a78d6"
C_F2 = "#008300"

def f1(x, y):
    """First objective function."""
    return x ** 2 * y

def f2(x, y):
    """Second objective function — bounded, anisotropic."""
    return np.log(1.0 + x ** 2) + 2.0 * np.log(1.0 + y ** 2)

def grad_f1(x, y):
    """Gradient of f1."""
    return np.array([2.0 * x * y, x ** 2])

def coupled_descent(x0, y0, eta, n_steps, period):
    """
    Genuine coupled gradient descent on f1.
    Returns arrays of f1-values, f2-values, and the empirical C.
    """
    x, y = float(x0), float(y0)
    vals1 = np.empty(n_steps)
    vals2 = np.empty(n_steps)
    swaps = np.zeros(n_steps, dtype=np.int8)

    for i in range(n_steps):
        vals1[i] = f1(x, y)
        vals2[i] = f2(x, y)

        gx, gy = grad_f1(x, y)
        dx = -eta * gx
        dy = -eta * gy

        swap = (period > 0) and ((i + 1) % period == 0)
        swaps[i] = int(swap)

        if swap:
            x += dy
            y += dx
        else:
            x += dx
            y += dy

        if not (np.isfinite(x) and np.isfinite(y)):
            raise FloatingPointError(f"non-finite state at step {i+1}")

    return vals1, vals2, float(swaps.mean())


def make_figure(x0, y0, eta, n_steps, filename):
    """
    4-panel figure: C = 0, 0.25, 0.50, 1.00.
    Each panel plots the normalized outputs of f1 and f2 along the
    coupled trajectory driven by grad(f1).
    """
    configs = [
        (0.00, 0, "no swaps"),
        (0.25, 4, "swap every 4th step"),
        (0.50, 2, "swap every 2nd step"),
        (1.00, 1, "swap every step"),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), sharey=False)
    fig.patch.set_facecolor(SURFACE)
    iterations = np.arange(1, n_steps + 1)

    for ax, (C, period, desc) in zip(axes, configs):
        v1, v2, actual_C = coupled_descent(x0, y0, eta, n_steps, period)

        # Normalise so both curves start at 1.0 — this keeps them on the
        # same scale and makes the coupling-induced divergence visible.
        v1_norm = v1 / v1[0]
        v2_norm = v2 / v2[0]

        ax.set_facecolor(SURFACE)
        ax.plot(iterations, v1_norm, color=C_F1, lw=2, label=r"$f_1 / f_{1,0}$")
        ax.plot(iterations, v2_norm, color=C_F2, lw=2, label=r"$f_2 / f_{2,0}$")
        ax.axhline(1.0, color=GRID, lw=0.9, ls="--")
        ax.axhline(0.0, color=GRID, lw=0.9)

        ax.set_title(
            f"$C = {C}$\n(actual $C = {actual_C:.2f}$)",
            fontsize=11, color=TEXT_PRIMARY, pad=10,
        )
        ax.text(
            0.5, 1.005, desc, transform=ax.transAxes,
            ha="center", va="bottom", fontsize=8.5, color=TEXT_SECONDARY,
        )
        ax.grid(True, color=GRID, lw=0.8)
        ax.set_axisbelow(True)

        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        for spine in ("left", "bottom"):
            ax.spines[spine].set_color(GRID)
        ax.tick_params(colors=TEXT_SECONDARY, labelsize=8)
        ax.set_xlabel("iteration $i$", fontsize=10, color=TEXT_SECONDARY)
        ax.set_xlim(1, n_steps)

    axes[0].set_ylabel(
        "normalised function value", fontsize=10, color=TEXT_SECONDARY
    )

    fig.legend(
        handles=[
            plt.Line2D([], [], color=C_F1, lw=2, label=r"$f_1(x_i,y_i)\,/\,f_1(x_0,y_0)$"),
            plt.Line2D([], [], color=C_F2, lw=2, label=r"$f_2(x_i,y_i)\,/\,f_2(x_0,y_0)$"),
        ],
        loc="upper right", ncol=2, frameon=False, fontsize=10,
        bbox_to_anchor=(0.99, 1.02), labelcolor=TEXT_PRIMARY,
    )
    fig.suptitle(
        "Coupled trajectories: normalised outputs of $f_1$ and $f_2$",
        x=0.01, ha="left", fontsize=13, color=TEXT_PRIMARY,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.90])

    out = Path(filename)
    fig.savefig(out, dpi=160, facecolor=SURFACE, bbox_inches="tight")
    print(f"Saved {out}")
    plt.close(fig)


# --- generate ---
make_figure(
    x0=1.0, y0=2.0, eta=0.1, n_steps=40,
    filename="coupling-f1-f2-outputs.png",
)

Saved coupling-f1-f2-outputs.png
